# 00 — Data Audit & Scope Lock (Milestone 0)

*Part of **Gamaka Floor** — a raga pitch-contour study.*

Inventory and format verification for the Saraga Carnatic 1.5 annotation corpus.
**No modelling here.** Outputs: coverage tables, the section-file merge answer, the
allied-pair search, and headline numbers written to `artifacts/`.

See `CLAUDE.md` (facts/decisions) and `spec.md` (process). Data facts are verified
against disk in this notebook, not assumed.

In [1]:
import json, glob, os, sys
import numpy as np
import pandas as pd

# --- DATA_ROOT switch: local Windows vs Kaggle mount ---
CANDIDATES = [
    r"D:\sg\saraga1.5_carnatic",                 # local
    "/kaggle/input/saraga-carnatic-annotations",   # Kaggle (adjust to actual mount)
]
DATA_ROOT = next((p for p in CANDIDATES if os.path.isdir(p)), None)
assert DATA_ROOT, f"No DATA_ROOT found among {CANDIDATES}"
ART = os.path.join(os.path.dirname(os.getcwd()), "artifacts") if os.path.basename(os.getcwd())=="notebooks" \
      else os.path.join(os.getcwd(), "artifacts")
os.makedirs(os.path.join(ART, "figures"), exist_ok=True)
print("DATA_ROOT =", DATA_ROOT)
print("ARTIFACTS =", ART)
pd.set_option("display.width", 180); pd.set_option("display.max_rows", 300)

DATA_ROOT = D:\sg\saraga1.5_carnatic
ARTIFACTS = D:\DGM\artifacts


## 1. JSON metadata keys (verify against a real file)

In [2]:
metas = sorted(glob.glob(os.path.join(DATA_ROOT, "**", "*.json"), recursive=True))
print(len(metas), "metadata json files")
# find first with a non-empty raaga to show the shape
def raga_of(m):   return (m.get("raaga") or [{}])[0].get("name")
def artist_of(m): return (m.get("album_artists") or [{}])[0].get("name")
ex = next(json.load(open(p, encoding="utf-8")) for p in metas
          if raga_of(json.load(open(p, encoding="utf-8"))))
print("top-level keys:", list(ex.keys()))
print("raaga[0].name        ->", raga_of(ex))
print("album_artists[0].name ->", artist_of(ex))
print("NOTE: 'raaga' can be an EMPTY list (not all recordings are raga-labelled).")
print("NOTE: 'artists' is nested as artists[0]['artist']['name']; use album_artists.")

249 metadata json files
top-level keys: ['mbid', 'title', 'length', 'artists', 'raaga', 'taala', 'form', 'work', 'concert', 'album_artists']
raaga[0].name        -> Kāmavardani/Pantuvarāḷi
album_artists[0].name -> Akkarai Sisters
NOTE: 'raaga' can be an EMPTY list (not all recordings are raga-labelled).
NOTE: 'artists' is nested as artists[0]['artist']['name']; use album_artists.


## 2. File inventory

In [3]:
def has(folder, pat): return bool(glob.glob(os.path.join(folder, pat)))
recs = []
for meta in metas:
    folder = os.path.dirname(meta)
    try: m = json.load(open(meta, encoding="utf-8"))
    except Exception as e:
        print("BAD JSON:", meta, e); continue
    recs.append(dict(
        folder=folder, base=os.path.basename(folder),
        raga=raga_of(m), artist=artist_of(m),
        has_p=has(folder, "*.sections-manual-p.txt"),
        has_plain=has(folder, "*.sections-manual.txt"),
        has_pitch=has(folder, "*.pitch.txt"),
        has_vocpitch=has(folder, "*.pitch-vocal.txt"),
        has_tonic=has(folder, "*.ctonic.txt"),
        has_mphrases=has(folder, "*.mphrases-manual.txt")))
R = pd.DataFrame(recs)
print(f"metadata json:      {len(R)}")
print(f"raga-labelled:      {R.raga.notna().sum()}")
print(f"pitch.txt:          {R.has_pitch.sum()}   pitch-vocal.txt: {R.has_vocpitch.sum()}   ctonic.txt: {R.has_tonic.sum()}")
print(f"sections -p:        {R.has_p.sum()}   sections plain: {R.has_plain.sum()}   mphrases: {R.has_mphrases.sum()}")
print(f"sections BOTH:      {(R.has_p & R.has_plain).sum()}   EITHER: {(R.has_p|R.has_plain).sum()}   NEITHER: {(~R.has_p & ~R.has_plain).sum()}")
print(f"raga + pitch + tonic (quantisation-usable): {(R.raga.notna() & R.has_pitch & R.has_tonic).sum()}")

metadata json:      249
raga-labelled:      184
pitch.txt:          197   pitch-vocal.txt: 56   ctonic.txt: 197
sections -p:        119   sections plain: 73   mphrases: 117
sections BOTH:      73   EITHER: 119   NEITHER: 130
raga + pitch + tonic (quantisation-usable): 184


## 3. Section files: helpers + the MERGE question

`*.sections-manual*.txt` are **4 tab columns**: `start, 1, DURATION, label`. Column 3
is duration, not end. Alapana = label containing `ālāp` and not `violin`.

In [4]:
def read_sections(path):
    out = []
    for line in open(path, encoding="utf-8"):
        p = line.rstrip("\n").split("\t")
        if len(p) < 4: continue
        try: s, d = float(p[0]), float(p[2])
        except ValueError: continue
        out.append((s, d, p[3].strip()))
    return out

def is_alap(lab):
    l = lab.lower(); return ("ālāp" in l or "alap" in l) and "violin" not in l

def is_vocal(lab):
    l = lab.lower(); return ("violin" not in l) and ("tani" not in l) and ("tāni" not in l)

def merge_intervals(ivs):
    if not ivs: return 0.0
    ivs = sorted(ivs); tot = 0.0; cs, ce = ivs[0]
    for s, e in ivs[1:]:
        if s <= ce: ce = max(ce, e)
        else: tot += ce - cs; cs, ce = s, e
    return tot + (ce - cs)

def sfile(folder, kind):  # kind in {'p','plain'}
    g = glob.glob(os.path.join(folder, f"*.sections-manual-p.txt")) if kind=="p" \
        else glob.glob(os.path.join(folder, f"*.sections-manual.txt"))
    return g[0] if g else None

In [5]:
rows = []
for _, r in R[R.raga.notna()].iterrows():
    fp, fpl = sfile(r.folder, "p"), sfile(r.folder, "plain")
    a_p  = [(s, s+d) for s,d,l in read_sections(fp)  if is_alap(l)] if fp  else []
    a_pl = [(s, s+d) for s,d,l in read_sections(fpl) if is_alap(l)] if fpl else []
    rows.append(dict(raga=r.raga, artist=r.artist, base=r.base,
                     alap_p=merge_intervals(a_p)/60,
                     alap_plain=merge_intervals(a_pl)/60,
                     alap_merged=merge_intervals(a_p + a_pl)/60))
A = pd.DataFrame(rows)
print("Total ALAPANA minutes (section-duration) by source:")
print(f"  -p only:  {A.alap_p.sum():6.1f} min across {(A.alap_p>0).sum()} recordings")
print(f"  plain:    {A.alap_plain.sum():6.1f} min across {(A.alap_plain>0).sum()} recordings")
print(f"  MERGED:   {A.alap_merged.sum():6.1f} min across {(A.alap_merged>0).sum()} recordings")
gain = A[(A.alap_merged - A.alap_p) > 0.05]
only_plain = A[(A.alap_p < 0.05) & (A.alap_plain >= 0.05)]
print(f"\nrecordings where merge beats -p alone: {len(gain)}")
print(f"recordings with alapana in plain but not -p: {len(only_plain)}")
print("\nVERDICT: plain-section recordings are a strict subset of -p, and -p covers all")
print("their alapana. Merging adds nothing. => USE -p ONLY. (Merge question resolved.)")

Total ALAPANA minutes (section-duration) by source:
  -p only:   252.6 min across 42 recordings
  plain:     143.8 min across 25 recordings
  MERGED:    252.6 min across 42 recordings

recordings where merge beats -p alone: 0
recordings with alapana in plain but not -p: 0

VERDICT: plain-section recordings are a strict subset of -p, and -p covers all
their alapana. Merging adds nothing. => USE -p ONLY. (Merge question resolved.)


## 4. Section-label vocabulary

In [6]:
labrows = []
for _, r in R.iterrows():
    fp = sfile(r.folder, "p") or sfile(r.folder, "plain")
    if not fp: continue
    for s,d,l in read_sections(fp): labrows.append(dict(label=l, dur=d))
L = pd.DataFrame(labrows)
vocab = (L.groupby("label").agg(n=("dur","count"), minutes=("dur", lambda x: x.sum()/60))
           .sort_values("n", ascending=False))
print(vocab.head(20).round(1).to_string())
print(f"\n'Vocal ālāp' total: {L[L.label.str.contains('ālāp') & ~L.label.str.contains('Violin')].dur.sum()/60:.1f} min")

                          n  minutes
label                               
Caraṇam                 147    258.3
Pallavi                 114    174.8
Anupallavi               84    123.6
Vocal ālāp               42    252.6
Kalpanā svara            34    200.4
Violin ālāp              28    144.3
Nereval                  18    112.5
Tani āvartana            11     98.9
Muktāyi svara            11     16.3
Ciṭṭa svara              11     23.4
-                         9     22.1
Tānam                     2     25.5
Caraṇam sahānā            2      3.2
Jati svara                2      2.2
Ślōka                     2      2.2
Verse                     2      1.4
Caraṇam yamuna kalyāṇi    1      1.5
Caraṇam sindhubhairavi    1      1.9
Caraṇam vasanta           1      1.3
Caraṇam hamīr kalyāṇi     1      2.4

'Vocal ālāp' total: 252.6 min


## 5. Alapana coverage per raga (merged sections; section-duration minutes)

In [7]:
Aa = A[A.alap_merged > 0]
alap_tbl = (Aa.groupby("raga")
              .agg(alap_min=("alap_merged","sum"), artists=("artist","nunique"), recs=("base","count"))
              .sort_values("alap_min", ascending=False))
print(alap_tbl.round(1).to_string())
print("\n>=15 min AND >=2 artists:")
print(alap_tbl.query("alap_min>=15 and artists>=2").round(1).to_string())
print("\nPer-artist alapana minutes (top ragas) — reveals single-artist domination:")
per_art = Aa[Aa.raga.isin(alap_tbl.query('alap_min>=8').index)]\
            .groupby(["raga","artist"]).alap_merged.sum().round(1)
print(per_art.to_string())
alap_tbl.round(3).to_csv(os.path.join(ART, "coverage_alapana.csv"))

                         alap_min  artists  recs
raga                                            
Karaharapriya                24.7        2     2
Mōhanaṁ                      18.9        2     2
Ṣanmukhapriya                17.7        2     2
Kāṁbhōji                     17.0        1     1
Bhairavi                     15.0        2     2
Harikāmbhōji                 13.3        2     2
Tōḍi                         12.8        3     3
Kāpi                         12.3        1     1
Sāvēri                       10.6        1     1
Kamās                        10.6        2     2
Kumudakriyā                  10.3        1     1
Simhēndra madhyamaṁ          10.1        1     1
Dhanyāsi                      9.2        1     1
Kāmavardani/Pantuvarāḷi       7.9        1     1
Latāngi                       7.6        1     1
Sārāmati                      7.6        1     1
Bēgaḍa                        7.4        2     2
Kēdāragauḷa                   7.1        1     1
Rītigauḷa           

## 6. Quantisation inventory (all raga-labelled recordings with pitch + tonic)

In [8]:
Q = R[R.raga.notna() & R.has_pitch & R.has_tonic]
qtbl = (Q.groupby("raga").agg(recs=("base","count"), artists=("artist","nunique"))
          .sort_values(["recs","artists"], ascending=False))
usable = qtbl.query("recs>=2 and artists>=2")
print(f"usable recordings: {len(Q)}   distinct ragas: {Q.raga.nunique()}")
print(f"ragas with >=2 recs & >=2 artists: {len(usable)}")
print(f"ragas with >=3 recs & >=2 artists: {qtbl.query('recs>=3 and artists>=2').shape[0]}")
print(f"ragas with >=3 recs & >=3 artists: {qtbl.query('recs>=3 and artists>=3').shape[0]}")
print("\nTop of the usable set:")
print(usable.head(25).to_string())
qtbl.to_csv(os.path.join(ART, "coverage_quantisation.csv"))

usable recordings: 184   distinct ragas: 96
ragas with >=2 recs & >=2 artists: 42
ragas with >=3 recs & >=2 artists: 19
ragas with >=3 recs & >=3 artists: 16

Top of the usable set:
                 recs  artists
raga                          
Rāgamālika          8        7
Tōḍi                7        7
Saurāṣtraṁ          7        5
Kamās               7        4
Behāg               5        4
Bhairavi            5        4
Mōhanaṁ             4        4
Rītigauḷa           4        4
Suraṭi              4        4
Kalyāṇi             4        3
Ṣanmukhapriya       4        3
Bēgaḍa              3        3
Harikāmbhōji        3        3
Jōnpuri             3        3
Sindhubhairavi      3        3
Sāvēri              3        3
Kānaḍa              3        2
Kāṁbhōji            3        2
Śankarābharaṇaṁ     3        2
Amṛtavarṣiṇi        2        2
Gauḷa               2        2
Hamīr kaḷyaṇi       2        2
Jaganmōhini         2        2
Kalgaḍa             2        2
Karaharapriy

## 7. Per-artist vocal minutes + allied-pair search

In [9]:
vrows = []
for _, r in Q.iterrows():
    fp = sfile(r.folder, "p") or sfile(r.folder, "plain")
    voc = sum(d for s,d,l in read_sections(fp) if is_vocal(l))/60 if fp else 0.0
    alp = sum(d for s,d,l in read_sections(fp) if is_alap(l))/60 if fp else 0.0
    vrows.append(dict(raga=r.raga, artist=r.artist, base=r.base, voc_min=voc, alap_min=alp, has_sections=bool(fp)))
V = pd.DataFrame(vrows)

# Musically-allied candidate pairs (shared/adjacent note-set, differ in gamaka/prayoga)
CANDS = [
 ("Suraṭi / Kēdāragauḷa",       ["Suraṭi","Kēdāragauḷa"],          "classic allied, both janya of Harikambhoji"),
 ("Kāṁbhōji / Harikāmbhōji",    ["Kāṁbhōji","Harikāmbhōji"],       "janya vs parent, shared body"),
 ("Ābhōgi / Śrīranjani",        ["Ābhōgi","Śrīranjani"],           "Sriranjani = Abhogi + N2"),
 ("Bēgaḍa / Śankarābharaṇaṁ",   ["Bēgaḍa","Śankarābharaṇaṁ"],      "Begada janya of Sankarabharanam"),
 ("Kalyāṇi / Hamīr kaḷyaṇi",    ["Kalyāṇi","Hamīr kaḷyaṇi"],       "Hamir Kalyani related to Kalyani"),
]
for name, ragas, note in CANDS:
    print(f"\n--- {name}   ({note}) ---")
    sub = V[V.raga.isin(ragas)]
    for rg in ragas:
        g = sub[sub.raga==rg]
        strong = (g.groupby('artist').voc_min.sum()>=4).sum()
        print(f"  {rg:<16} vocal={g.voc_min.sum():5.1f}m  alapana={g.alap_min.sum():5.1f}m  "
              f"artists={g.artist.nunique()}  artists>=4m={strong}")
        for a,v in g.groupby('artist').voc_min.sum().sort_values(ascending=False).round(1).items():
            print(f"        {a:<32}{v:5.1f}m")
    shared = sorted(set(sub[sub.raga==ragas[0]].artist) & set(sub[sub.raga==ragas[1]].artist))
    print(f"  shared artists: {shared or 'NONE'}")


--- Suraṭi / Kēdāragauḷa   (classic allied, both janya of Harikambhoji) ---
  Suraṭi           vocal=  7.6m  alapana=  0.3m  artists=4  artists>=4m=1
        Ashwath Narayanan                 7.1m
        Sumithra Vasudev                  0.5m
        Mahati                            0.0m
        S Sundar                          0.0m
  Kēdāragauḷa      vocal= 35.3m  alapana=  7.1m  artists=2  artists>=4m=2
        Rithvik Raja                     25.4m
        Cherthala Ranganatha Sharma       9.9m
  shared artists: NONE

--- Kāṁbhōji / Harikāmbhōji   (janya vs parent, shared body) ---
  Kāṁbhōji         vocal= 58.7m  alapana= 17.0m  artists=2  artists>=4m=1
        Sanjay Subrahmanyan              58.7m
        Chaitra Sairam                    0.0m
  Harikāmbhōji     vocal= 50.2m  alapana= 13.3m  artists=3  artists>=4m=2
        Sanjay Subrahmanyan              38.5m
        V. Shankaranarayanan             11.7m
        S Sundar                          0.0m
  shared artists: ['S

## 8. Headline numbers -> artifacts/numbers.json

Collect M0 facts so later notebooks and the report read numbers instead of re-deriving.

In [10]:
numbers = {
  "m0": {
    "n_metadata_json": int(len(R)),
    "n_raga_labelled": int(R.raga.notna().sum()),
    "n_pitch": int(R.has_pitch.sum()),
    "n_pitch_vocal": int(R.has_vocpitch.sum()),
    "n_tonic": int(R.has_tonic.sum()),
    "n_sections_p": int(R.has_p.sum()),
    "n_sections_plain": int(R.has_plain.sum()),
    "sections_plain_is_subset_of_p": bool((R.has_plain & ~R.has_p).sum()==0),
    "merge_adds_recordings": int(len(A[(A.alap_p<0.05)&(A.alap_plain>=0.05)])),
    "alap_min_p_only": round(float(A.alap_p.sum()),1),
    "alap_min_merged": round(float(A.alap_merged.sum()),1),
    "n_quant_usable_recordings": int(len(Q)),
    "n_quant_distinct_ragas": int(Q.raga.nunique()),
    "n_ragas_ge2rec_ge2artist": int(len(usable)),
    "alap_balanced_ragas": ["Karaharapriya", "Mōhanaṁ"],
    "allied_pair_found": False,
    "decision_status": "AWAITING USER SCOPE CONFIRMATION (see status/STATUS_M0.md)",
  }
}
outp = os.path.join(ART, "numbers.json")
if os.path.exists(outp):
    with open(outp, encoding="utf-8") as f: existing = json.load(f)
    existing.update(numbers); numbers = existing
with open(outp, "w", encoding="utf-8") as f: json.dump(numbers, f, ensure_ascii=False, indent=2)
print("wrote", outp)
print(json.dumps(numbers["m0"], ensure_ascii=False, indent=2))

wrote D:\DGM\artifacts\numbers.json
{
  "n_metadata_json": 249,
  "n_raga_labelled": 184,
  "n_pitch": 197,
  "n_pitch_vocal": 56,
  "n_tonic": 197,
  "n_sections_p": 119,
  "n_sections_plain": 73,
  "sections_plain_is_subset_of_p": true,
  "merge_adds_recordings": 0,
  "alap_min_p_only": 252.6,
  "alap_min_merged": 252.6,
  "n_quant_usable_recordings": 184,
  "n_quant_distinct_ragas": 96,
  "n_ragas_ge2rec_ge2artist": 42,
  "alap_balanced_ragas": [
    "Karaharapriya",
    "Mōhanaṁ"
  ],
  "allied_pair_found": false,
  "decision_status": "AWAITING USER SCOPE CONFIRMATION (see status/STATUS_M0.md)"
}
